In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [4]:

#model = "claude-haiku-4-5"
#model = "claude-sonnet-4-6"
#model = "claude-haiku-4-5-20251001"

In [2]:
def add_user_messages(messages, text):
    user_message  = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_messages(messages, text):
    assistant_message  = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_messages(messages, prompt)
    add_assistant_messages(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [5]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [6]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:
{test_case["task"]}
"""
    messages = []
    add_user_messages(messages, prompt)
    output = chat(messages)
    return output

In [7]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TO DO
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [8]:
def run_eval(dataset):
    """Loads the datset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [11]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [13]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Name Validator\n\nHere's a comprehensive solution with regex patterns and validation logic:\n\n```regex\n^[a-z0-9]([a-z0-9\\-\\.]*[a-z0-9])?$\n```\n\nHowever, this basic pattern doesn't enforce all AWS rules. Here's a **complete solution** with full validation:\n\n## Complete Regex Pattern\n\n```regex\n^(?!.*\\.\\.)[a-z0-9]([a-z0-9\\-\\.]{1,61}[a-z0-9])?$\n```\n\n## Breaking Down the Pattern\n\n```\n^                    # Start of string\n(?!.*\\.\\.)          # Negative lookahead: no consecutive periods\n[a-z0-9]            # Must start with letter or number\n(\n  [a-z0-9\\-\\.]{1,61}  # Middle: 1-61 chars (letters, numbers, hyphens, periods)\n  [a-z0-9]            # Must end with letter or number\n)?                   # Middle part optional (for 3-char buckets)\n$                    # End of string\n```\n\n## Implementation Examples\n\n### JavaScript\n```javascript\nconst s3BucketRegex = /^(?!.*\\.\\.)[a-z0-9]([a-z0-9\\-\\.]{1,61}[a-z0-9])?$/;\n\n